# 🌸 Clasificador de Iris con Streamlit y Localtunnel (funcional en Colab)

In [4]:
# Instala Streamlit y scikit-learn mediante pip.
# - streamlit: framework para crear la interfaz web interactiva.
# - scikit-learn: librería para el modelo (RandomForest y dataset Iris).
# El flag -q suprime salida verborreica para mantener la celda más limpia.
!pip install streamlit scikit-learn -q

# Descargar el binario oficial de cloudflared (cliente de Cloudflare Tunnel).
# - wget descarga el archivo desde la última release del repositorio GitHub.
# - -q: modo silencioso; -O /usr/local/bin/cloudflared: escribe el binario en /usr/local/bin
# Esto coloca el ejecutable en una ruta del sistema para poder invocarlo como comando.
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

# Dar permiso de ejecución al binario descargado.
# - chmod +x hace que el archivo sea ejecutable por el sistema.
!chmod +x /usr/local/bin/cloudflared

# Verificar que cloudflared se instaló correctamente mostrando su versión.
# Si el comando devuelve la versión, significa que el binario está disponible en PATH.
!cloudflared --version




cloudflared version 2025.11.1 (built 2025-11-07-16:59 UTC)


In [5]:
# ==========================================
# Creamos el archivo principal de la aplicación Streamlit
# ==========================================
%%writefile app.py

# Importación de librerías necesarias
import streamlit as st                                   # Para la interfaz web interactiva
from sklearn.datasets import load_iris                   # Dataset Iris incorporado en scikit-learn
from sklearn.ensemble import RandomForestClassifier      # Modelo de clasificación utilizado
from sklearn.model_selection import train_test_split     # División de datos en entrenamiento y prueba
from sklearn.metrics import accuracy_score               # Métrica de exactitud del modelo

# --- Configuración general de la página ---
# Define el título, ícono y disposición de la interfaz de Streamlit.
st.set_page_config(page_title="Clasificador Iris", page_icon="🌸", layout="centered")

# --- Encabezado e introducción ---
st.title("🌿 Clasificador de flores Iris")
st.markdown(
    "Modelo de clasificación basado en **Random Forest**, "
    "entrenado con el dataset *Iris* de *scikit-learn*."
)

# --- Cargar dataset y entrenar el modelo ---
# Cargamos el conjunto de datos Iris, que contiene 150 muestras y 4 características.
iris = load_iris()
X, y = iris.data, iris.target
class_names = iris.target_names  # Nombres de las especies de flores

# División del dataset en 80% para entrenamiento y 20% para prueba.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Creación y entrenamiento del modelo Random Forest.
# n_estimators define la cantidad de árboles de decisión.
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Cálculo de la exactitud del modelo sobre los datos de prueba.
accuracy = accuracy_score(y_test, model.predict(X_test))
st.info(f"Exactitud del modelo: {accuracy:.2%}")

# --- Entradas del usuario mediante sliders ---
# Cada slider permite ajustar el valor de una característica de la flor.
st.subheader("🔢 Ajusta las medidas de la flor")
sepal_length = st.slider("Longitud del sépalo (cm)", 4.0, 8.0, 5.1, 0.1)
sepal_width  = st.slider("Ancho del sépalo (cm)", 2.0, 4.5, 3.5, 0.1)
petal_length = st.slider("Longitud del pétalo (cm)", 1.0, 7.0, 1.4, 0.1)
petal_width  = st.slider("Ancho del pétalo (cm)", 0.1, 2.5, 0.2, 0.1)

# --- Predicción del modelo ---
# Cuando el usuario pulsa el botón, se genera una predicción con los valores seleccionados.
if st.button("Predecir 🌸"):
    # Vector de características con los valores elegidos
    features = [[sepal_length, sepal_width, petal_length, petal_width]]

    # Realizamos la predicción y obtenemos las probabilidades por clase.
    prediction = model.predict(features)[0]
    proba = model.predict_proba(features)[0]

    # Mostramos el resultado principal y las probabilidades de cada especie.
    st.success(f"**Predicción:** {class_names[prediction].capitalize()}")
    st.subheader("📊 Probabilidades:")
    for i, name in enumerate(class_names):
        st.write(f"- {name.capitalize()}: {proba[i]*100:.2f}%")




Overwriting app.py


In [ ]:
# Importamos módulos necesarios para ejecutar procesos en background y gestionar esperas.
import threading       # Para ejecutar el servidor Streamlit en un hilo separado (no bloqueante)
import subprocess      # Para lanzar comandos del sistema (ej. 'streamlit run')
import time            # Para pausar la ejecución y dar tiempo a que el servidor arranque

# Definimos una función que inicia el servidor Streamlit como si la ejecutáramos en la terminal:
def run_app():
    # subprocess.run ejecuta el comando tal cual en la shell.
    # ["streamlit", "run", "app.py", "--server.port", "8501"]
    # - "streamlit run app.py": arranca la aplicación definida en app.py
    # - "--server.port 8501": fuerza el puerto 8501 (por defecto de Streamlit)
    subprocess.run(["streamlit", "run", "app.py", "--server.port", "8501"])

# Lanzamos la función run_app en un hilo (thread) en modo daemon:
# - daemon=True: asegura que el hilo no impida que el proceso principal termine si fuese necesario.
# - Al ejecutarlo en segundo plano, la celda no queda bloqueada y podemos seguir con la creación del túnel.
thread = threading.Thread(target=run_app, daemon=True)
thread.start()   # Inicia el hilo que ejecutará Streamlit

# Esperamos unos segundos para que el servidor Streamlit termine de iniciarse completamente.
# - Si se conecta el túnel antes de que el servidor esté listo, la URL pública puede no responder.
# - Ajuste el tiempo si su entorno es más lento (p. ej., time.sleep(12)).
time.sleep(8)

# Crear el túnel público con cloudflared para exponer http://localhost:8501 al exterior.
# - cloudflared tunnel --url http://localhost:8501: crea un "quick tunnel" y devuelve una URL trycloudflare.com
# - --no-autoupdate: evita que cloudflared intente actualizarse automáticamente (útil en entornos temporales como Colab)
# NOTA: Esta línea debe ejecutarse en una celda de Colab (prefijada con '!') para invocar el comando del sistema.
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate



2025-11-09T17:04:35Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2025-11-09T17:04:35Z INF Requesting new quick Tunnel on trycloudflare.com...
2025-11-09T17:04:38Z INF +--------------------------------------------------------------------------------------------+
2025-11-09T17:04:38Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2025-11-09T17:04:38Z INF |  https://arnold-pressed-ranks-architects.trycloudflare